# polygraphics-backend — free GPU demo on Google Colab

Spins up the full pipeline (SAM + DUSt3R + Gaussian Splatting + COLMAP fallback) on a free Colab T4 and exposes it on a public `trycloudflare.com` URL — **no accounts, no card, no tokens.**

## Before you click "Run all"
1. **Switch the runtime to GPU**: `Runtime > Change runtime type > T4 GPU`.
2. Then `Runtime > Run all`.

## What you get
- A public HTTPS URL that looks like `https://random-words-xyz.trycloudflare.com`
- Swagger at `<URL>/swagger`, health at `<URL>/health`
- DUSt3R + Gaussian Splatting at full GPU speed

## Free-tier limits to know
- ~12 h max session, ~90 min idle disconnect — keep this tab focused.
- All disk is wiped when the runtime stops. Download any `.glb` you want to keep.
- Re-running the **last cell** stops the previous server cleanly and reissues a fresh tunnel URL.

## 1. Verify GPU is attached

In [ ]:
!nvidia-smi || echo '\nNo GPU detected. Switch Runtime > Change runtime type > T4 GPU and try again.'

## 2. Clone the repository

In [ ]:
%%bash
set -e
REPO=/content/polygraphics-backend
if [ ! -d "$REPO" ]; then
  git clone https://github.com/mohannadfarhoud/polygraphics-backend.git "$REPO"
else
  cd "$REPO" && git pull --ff-only
fi
cd "$REPO" && git rev-parse --short HEAD

## 3. Install everything (DUSt3R, SAM, deps, checkpoints)

Takes ~3–5 minutes the first time. Re-running is fast (skips what's already there).

In [ ]:
%%bash
cd /content/polygraphics-backend
bash scripts/colab_setup.sh

## 4. (Optional) tweak runtime settings

Defaults are sensible. Uncomment to e.g. force the Gaussian-Splatting backend or raise iterations.

In [ ]:
import json
from pathlib import Path

p = Path('/content/polygraphics-backend/config/runtime_settings.json')
settings = json.loads(p.read_text())

# settings['reconstruction_backend'] = 'gaussian_splatting'  # force .ply path
# settings['gs_iterations'] = 7000                            # quick GS run
# settings['max_image_side'] = 1024

p.write_text(json.dumps(settings, indent=2))
print(json.dumps(settings, indent=2))

## 5. Start the API + open the public tunnel

**This cell stays running.** Watch the output for a line like:
```
https://random-words-xyz.trycloudflare.com
```
That's your public demo URL. Open `<URL>/swagger` in any browser to test.

Stop the cell to tear down the tunnel; re-run it to get a fresh URL.

In [ ]:
!bash /content/polygraphics-backend/scripts/colab_serve.sh